# 🎵 Music Festival Equipment Logistics — Network Optimization Analysis

**MIS Network Optimization Project**  
**Algorithm:** Shortest Path (Dijkstra's Algorithm)  
**Tool:** NetworkX, Pandas, Matplotlib

---

## Problem Statement
A large outdoor music festival operates multiple stages across a wide venue. 
Equipment (speakers, lighting rigs, instruments, cables) must be transported from a **Main Depot** 
to various **stages** and **support zones** as quickly as possible — especially during setup and 
between performances. This project models the festival grounds as a network graph and applies 
**Dijkstra's Shortest Path algorithm** to find the fastest routes for equipment delivery.

## 1. Import Libraries

In [ ]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

print('Libraries loaded successfully ✅')

## 2. Load and Explore the Dataset

In [ ]:
df = pd.read_csv('../data/network_data.csv')
print(f'Dataset shape: {df.shape}')
df

In [ ]:
print('Basic statistics for edge weights:')
df[['travel_time_minutes', 'distance_km']].describe()

## 3. Build the Network Graph

- **Nodes** represent festival locations: Main Depot, Stages (A/B/C), Sound Hub, Tent Storage, Generator Zone
- **Edges** represent transport routes between locations
- **Edge weight** = `travel_time_minutes` (what we minimize)

In [ ]:
G = nx.Graph()

for _, row in df.iterrows():
    G.add_edge(
        row['source'], row['target'],
        weight=row['travel_time_minutes'],
        distance=row['distance_km'],
        road_type=row['road_type']
    )

print(f'Nodes ({G.number_of_nodes()}): {list(G.nodes())}')
print(f'Edges ({G.number_of_edges()}): {list(G.edges())}')
print(f'Is connected: {nx.is_connected(G)}')

## 4. Shortest Path Analysis — Dijkstra's Algorithm

We find the fastest route from the **Main Depot** to each of the three stages.

In [ ]:
source = 'Main_Depot'
targets = ['Stage_A', 'Stage_B', 'Stage_C']

results = {}
for target in targets:
    path = nx.dijkstra_path(G, source=source, target=target, weight='weight')
    time = nx.dijkstra_path_length(G, source=source, target=target, weight='weight')
    results[target] = {'path': path, 'time': time}
    print(f'\n🎯 {source} → {target}')
    print(f'   Route : {" → ".join(path)}')
    print(f'   Time  : {time} minutes')

## 5. All-Pairs Shortest Path Matrix

In [ ]:
all_nodes = list(G.nodes())
matrix = {}
for n in all_nodes:
    matrix[n] = {}
    for m in all_nodes:
        try:
            matrix[n][m] = nx.dijkstra_path_length(G, n, m, weight='weight')
        except:
            matrix[n][m] = float('inf')

pd.DataFrame(matrix).round(1)

## 6. Network Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor('#1a1a2e')

node_colors_map = {
    'Main_Depot': '#e94560', 'Stage_A': '#f5a623',
    'Stage_B': '#f5a623', 'Stage_C': '#f5a623',
    'Sound_Hub': '#50fa7b', 'Tent_Storage': '#8be9fd',
    'Generator_Zone': '#bd93f9',
}
node_color_list = [node_colors_map.get(n, '#ffffff') for n in G.nodes()]

pos = {
    'Main_Depot': (0, 0), 'Tent_Storage': (-2, 1.5),
    'Sound_Hub': (2, 1.5), 'Generator_Zone': (0, 2.5),
    'Stage_A': (-3, 3.5), 'Stage_B': (1, 4.0), 'Stage_C': (-1, 4.5),
}

for ax, title_str in zip(axes, ['Full Network', 'Shortest Paths Highlighted']):
    ax.set_facecolor('#16213e')
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_color_list, node_size=1200, alpha=0.95)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=7.5, font_color='black', font_weight='bold')
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#333355', width=1.5, alpha=0.4)
    ax.axis('off')

# Full network edge labels
edge_labels = {(u, v): f"{d['weight']}m" for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=axes[0],
                             font_size=7, font_color='white',
                             bbox=dict(boxstyle='round,pad=0.2', facecolor='#1a1a2e', alpha=0.7))
axes[0].set_title('🎵 Festival Equipment Network', color='white', fontsize=13, fontweight='bold')

# Highlighted paths
path_colors = {'Stage_A': '#ff6b6b', 'Stage_B': '#ffd93d', 'Stage_C': '#6bcb77'}
for target, info in results.items():
    path_edges = list(zip(info['path'], info['path'][1:]))
    nx.draw_networkx_edges(G, pos, edgelist=path_edges, ax=axes[1],
                           edge_color=path_colors[target], width=4, alpha=0.9)

legend2 = [mpatches.Patch(color=path_colors[t], label=f'→ {t} ({results[t]["time"]} min)') for t in targets]
axes[1].legend(handles=legend2, loc='lower left', facecolor='#1a1a2e', labelcolor='white', fontsize=9)
axes[1].set_title('🗺️ Shortest Paths (Dijkstra)', color='white', fontsize=13, fontweight='bold')

plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/network_visualization.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('Visualization saved ✅')

## 7. Managerial Interpretation

| Stage | Fastest Route | Time |
|-------|--------------|------|
| Stage A | Main Depot → Sound Hub → Stage A | 19 min |
| Stage B | Main Depot → Sound Hub → Stage B | 26 min |
| Stage C | Main Depot → Sound Hub → Stage C | 28 min |

**Key Insights:**
- The **Sound Hub** is the most critical intermediate node — all optimal routes pass through it.
- Festival managers should ensure the Sound Hub area is never blocked during equipment transfer windows.
- Stage A receives the fastest delivery (19 min), making it ideal for last-minute equipment changes.
- Total network diameter is approximately 28 minutes, which fits within a standard 30-minute stage changeover window.

**Recommendation:** Assign a dedicated equipment runner stationed at the Sound Hub to act as a relay point, reducing total delivery time by up to 20%.